# Notebook 20. Eastern Siberian High Position and Cleaned-Cluster Association

This notebook follows the cleaned `k = 2` subtype framework and asks a new physical question:

**Are cleaned `Cluster 1` and cleaned `Cluster 2` associated with different Eastern Siberian High positions or strengths?**

Primary goals:

- use the cleaned `k = 2` labels from `Notebook 15`
- use `mean sea-level pressure` (`msl`) anomaly instead of low-level pressure-level fields
- locate the peak positive `msl` anomaly inside a broad Eastern Siberian High search box for each event
- compare high-center longitude, latitude, and anomaly magnitude by cleaned cluster

Secondary overlay:

- if `Notebook 18` quartile selection is available, compare the Eastern Siberian High metrics across the `lower / middle / upper` subsets of cleaned `Cluster 1`
- keep this as a diagnostic, not as a new clustering variable


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import xarray as xr

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import BoundingBox
from jpcz_catalog.diagnostics import load_snapshot
from jpcz_catalog.era5 import open_arco_era5

CLEANED_RUN_EXPORT_DIR = Path("outputs/verification/objective_subtype_low_level_cleaned_sensitivity")
QUARTILE_EXPORT_DIR = Path("outputs/verification/objective_subtype_cleaned_cluster_quartiles")
HIGH_EXPORT_DIR = Path("outputs/verification/objective_subtype_siberian_high_association")
HIGH_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_subtype_siberian_high_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CLUSTERED_EVENTS_PATH = CLEANED_RUN_EXPORT_DIR / "clustered_events_cleaned_low_level_k2_k3_k4.csv"
NOTEBOOK15_SOLUTION_SUMMARY_PATH = CLEANED_RUN_EXPORT_DIR / "cleaned_low_level_solution_summary.csv"
QUARTILE_SELECTION_PATH = QUARTILE_EXPORT_DIR / "cleaned_k2_cluster1_quartile_selection.csv"
MSL_CLIMATOLOGY_PATH = HIGH_EXPORT_DIR / "msl_ndjf_monthly_climatology_siberian_domain.nc"
EVENT_STACK_PATH = HIGH_EXPORT_DIR / "cleaned_k2_siberian_high_event_anomaly_stack.nc"
EVENT_STACK_PARTIAL_PATH = HIGH_EXPORT_DIR / "cleaned_k2_siberian_high_event_anomaly_stack_partial.nc"
EVENT_METRICS_PATH = HIGH_EXPORT_DIR / "cleaned_k2_siberian_high_event_metrics.csv"
EVENT_METRICS_PARTIAL_PATH = HIGH_EXPORT_DIR / "cleaned_k2_siberian_high_event_metrics_partial.csv"
CLUSTER_COMPOSITE_PATH = HIGH_EXPORT_DIR / "cleaned_k2_siberian_high_cluster_composites.nc"
CLUSTER_SUMMARY_PATH = HIGH_EXPORT_DIR / "cleaned_k2_siberian_high_cluster_summary.csv"
POSITION_CROSSTAB_PATH = HIGH_EXPORT_DIR / "cleaned_k2_siberian_high_position_category_summary.csv"
QUARTILE_HIGH_SUMMARY_PATH = HIGH_EXPORT_DIR / "cleaned_k2_cluster1_siberian_high_quartile_summary.csv"
PLOT_INVENTORY_PATH = HIGH_EXPORT_DIR / "cleaned_k2_siberian_high_plot_inventory.csv"

PRIMARY_CLUSTER_COLUMN = "cleaned_cluster_ward_2"
EXPLORATORY_CLUSTER_COLUMN = "cleaned_cluster_ward_3"
PRIMARY_CLUSTER_A_ID = 1
PRIMARY_CLUSTER_B_ID = 2
ERA5_TIME_CHUNK = 48
CHECKPOINT_EVERY_EVENTS = 10
FORCE_REBUILD_HIGH_STACK = False
SAVE_PLOTS = True
NDJF_MONTH_ORDER = [11, 12, 1, 2]
MSL_VARIABLE = "mean_sea_level_pressure"

SIBERIAN_HIGH_PLOT_DOMAIN = BoundingBox(lon_min=100.0, lon_max=160.0, lat_min=35.0, lat_max=75.0)
SIBERIAN_HIGH_SEARCH_BOX = BoundingBox(lon_min=105.0, lon_max=155.0, lat_min=45.0, lat_max=72.0)
EAST_LONGITUDE_THRESHOLD = 130.0
SOUTH_LATITUDE_THRESHOLD = 55.0

FIELD_NAME = "msl_anomaly_peak_hpa"
FIELD_LABEL = "Mean sea-level pressure anomaly at event peak"
FIELD_UNITS = "hPa"


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None



def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True



def ordinal_word(value: int) -> str:
    lookup = {1: "first", 2: "second", 3: "third", 4: "fourth", 5: "fifth"}
    return lookup.get(value, f"{value}th")



def size_rank_descriptor(rank: int, total: int) -> str:
    if total <= 1:
        return "only subgroup"
    if rank == 1:
        return "largest subgroup"
    if rank == total:
        return "smallest subgroup"
    return f"{ordinal_word(rank)}-largest subgroup"



def build_cluster_labels_from_counts(cluster_counts: pd.Series | dict[int, int]):
    counts_dict = {int(cluster_id): int(n_events) for cluster_id, n_events in dict(cluster_counts).items()}
    ranked = sorted(counts_dict.items(), key=lambda item: (-item[1], item[0]))
    rank_lookup = {cluster_id: rank for rank, (cluster_id, _) in enumerate(ranked, start=1)}
    total = len(ranked)
    long_labels = {}
    rows = []
    for cluster_id, n_events in sorted(counts_dict.items()):
        descriptor = size_rank_descriptor(rank_lookup[cluster_id], total)
        long_labels[cluster_id] = f"Cluster {cluster_id}: n={n_events} ({descriptor})"
        rows.append(
            {
                "cluster_id": cluster_id,
                "n_events": n_events,
                "size_rank": rank_lookup[cluster_id],
                "size_descriptor": descriptor,
                "cluster_label": long_labels[cluster_id],
            }
        )
    return long_labels, pd.DataFrame(rows)



def month_label(month_number: int) -> str:
    lookup = {11: "Nov", 12: "Dec", 1: "Jan", 2: "Feb"}
    return lookup.get(int(month_number), str(month_number))



def load_cached_dataset(path: Path):
    if not path.exists():
        return None
    with xr.open_dataset(path) as ds:
        return ds.load()



def write_checkpoint_dataset(ds: xr.Dataset, path: Path):
    ds.sortby("event_index").to_netcdf(path)
    maybe_copy_to_drive(path)



def safe_concat(existing_ds: xr.Dataset | None, new_ds: xr.Dataset) -> xr.Dataset:
    if existing_ds is None:
        return new_ds
    return xr.concat([existing_ds, new_ds], dim="event_index").sortby("event_index")



def attach_event_metadata(field: xr.DataArray, row: pd.Series) -> xr.Dataset:
    event_index = int(row["event_index"])
    event_peak = np.datetime64(pd.Timestamp(row["event_peak"]).to_datetime64())
    expanded = field.expand_dims(event_index=[event_index]).to_dataset(name=FIELD_NAME)
    expanded = expanded.assign_coords(
        event_peak=("event_index", [event_peak]),
        cleaned_cluster_k2=("event_index", [int(row[PRIMARY_CLUSTER_COLUMN])]),
    )
    return expanded



def compute_monthly_msl_climatology(ds: xr.Dataset, *, years, months, domain) -> xr.DataArray:
    monthly_sums = {}
    monthly_counts = {}
    for month in months:
        month_sum = None
        month_count = 0
        for year in years:
            start = f"{year}-{month:02d}-01"
            end = (pd.Timestamp(start) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%dT23:00:00")
            print(f"Climatology month {month:02d}: loading {year}-{month:02d}")
            subset = ds[[MSL_VARIABLE]].sel(
                time=slice(start, end),
                longitude=slice(domain.lon_min, domain.lon_max),
                latitude=slice(domain.lat_max, domain.lat_min),
            )
            msl_hpa = (subset[MSL_VARIABLE] / 100.0).rename("msl_hpa")
            window_sum = msl_hpa.sum("time").load()
            window_count = int(msl_hpa.sizes["time"])
            month_sum = window_sum if month_sum is None else (month_sum + window_sum)
            month_count += window_count
        if month_sum is None or month_count == 0:
            continue
        monthly_sums[int(month)] = month_sum
        monthly_counts[int(month)] = month_count

    climatology_slices = []
    for month in sorted(monthly_sums):
        month_mean = (monthly_sums[month] / monthly_counts[month]).expand_dims(month=[month])
        climatology_slices.append(month_mean)
    climatology = xr.concat(climatology_slices, dim="month").rename("monthly_msl_climatology_hpa")
    climatology.attrs["units"] = "hPa"
    climatology.attrs["source_variable"] = MSL_VARIABLE
    return climatology



def load_or_update_msl_climatology(path: Path, *, years, months, domain, current_ds=None):
    if path.exists():
        climatology = xr.open_dataarray(path).load()
        cached_months = {int(month_value) for month_value in climatology["month"].values.tolist()}
    else:
        climatology = None
        cached_months = set()
    missing_months = sorted(set(months) - cached_months)
    if missing_months:
        print(f"Cached MSL climatology missing months: {missing_months}")
        print("Computing missing MSL climatology months one at a time and checkpointing each completed month to Drive.")
        current_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK}) if current_ds is None else current_ds
        for month in missing_months:
            month_climatology = compute_monthly_msl_climatology(
                current_ds,
                years=years,
                months=[month],
                domain=domain,
            )
            climatology = month_climatology if climatology is None else xr.concat([climatology, month_climatology], dim="month").sortby("month")
            climatology.to_netcdf(path)
            maybe_copy_to_drive(path)
            print(f"Checkpointed MSL climatology after month {month:02d}")
        climatology_source = "month-by-month checkpoints"
    else:
        climatology_source = "restored cached climatology"
    if climatology is None:
        raise RuntimeError("Unable to load or compute the MSL climatology.")
    return climatology, climatology_source, current_ds



def find_high_center(anomaly_field: xr.DataArray, search_box) -> dict[str, float]:
    subset = anomaly_field.sel(
        longitude=slice(search_box.lon_min, search_box.lon_max),
        latitude=slice(search_box.lat_max, search_box.lat_min),
    )
    values = np.asarray(subset.values, dtype=float)
    if values.size == 0 or not np.isfinite(values).any():
        return {
            "high_center_lon_degE": np.nan,
            "high_center_lat_degN": np.nan,
            "high_max_msl_anomaly_hpa": np.nan,
            "search_box_mean_msl_anomaly_hpa": np.nan,
            "positive_anomaly_fraction_in_search_box": np.nan,
        }
    max_index = np.unravel_index(np.nanargmax(values), values.shape)
    lat_idx, lon_idx = int(max_index[0]), int(max_index[1])
    return {
        "high_center_lon_degE": float(subset.longitude.values[lon_idx]),
        "high_center_lat_degN": float(subset.latitude.values[lat_idx]),
        "high_max_msl_anomaly_hpa": float(values[lat_idx, lon_idx]),
        "search_box_mean_msl_anomaly_hpa": float(np.nanmean(values)),
        "positive_anomaly_fraction_in_search_box": float(np.nanmean(values > 0.0)),
    }



def categorize_high_position(lon_degE: float, lat_degN: float) -> dict[str, object]:
    east_flag = bool(np.isfinite(lon_degE) and lon_degE >= EAST_LONGITUDE_THRESHOLD)
    south_flag = bool(np.isfinite(lat_degN) and lat_degN < SOUTH_LATITUDE_THRESHOLD)
    return {
        "high_center_east_of_130E": east_flag,
        "high_center_south_of_55N": south_flag,
        "high_position_category": f"{'east' if east_flag else 'west'}_{'south' if south_flag else 'north'}",
    }



def summarize_high_metrics(event_metrics_df: pd.DataFrame, cluster_label_lookup: dict[int, str]):
    summary_rows = []
    for cluster_id, subset in event_metrics_df.groupby(PRIMARY_CLUSTER_COLUMN):
        cluster_id = int(cluster_id)
        summary_rows.append(
            {
                "cluster_id": cluster_id,
                "cluster_label": cluster_label_lookup[cluster_id],
                "n_events": len(subset),
                "mean_high_center_lon_degE": float(subset["high_center_lon_degE"].mean()),
                "median_high_center_lon_degE": float(subset["high_center_lon_degE"].median()),
                "mean_high_center_lat_degN": float(subset["high_center_lat_degN"].mean()),
                "median_high_center_lat_degN": float(subset["high_center_lat_degN"].median()),
                "mean_high_max_msl_anomaly_hpa": float(subset["high_max_msl_anomaly_hpa"].mean()),
                "median_high_max_msl_anomaly_hpa": float(subset["high_max_msl_anomaly_hpa"].median()),
                "mean_search_box_mean_msl_anomaly_hpa": float(subset["search_box_mean_msl_anomaly_hpa"].mean()),
                "percent_events_high_center_east_of_130E": float(100.0 * subset["high_center_east_of_130E"].mean()),
                "percent_events_high_center_south_of_55N": float(100.0 * subset["high_center_south_of_55N"].mean()),
                "percent_events_positive_high_anomaly": float(100.0 * (subset["high_max_msl_anomaly_hpa"] > 0.0).mean()),
            }
        )
    cluster_summary_df = pd.DataFrame(summary_rows)

    category_order = ["west_north", "east_north", "west_south", "east_south"]
    count_table = pd.crosstab(event_metrics_df[PRIMARY_CLUSTER_COLUMN], event_metrics_df["high_position_category"])
    count_table = count_table.reindex(index=sorted(cluster_label_lookup), columns=category_order, fill_value=0)
    percent_table = count_table.div(count_table.sum(axis=1), axis=0).fillna(0.0) * 100.0
    crosstab_rows = []
    for cluster_id in count_table.index:
        for category in count_table.columns:
            crosstab_rows.append(
                {
                    "cluster_id": int(cluster_id),
                    "cluster_label": cluster_label_lookup[int(cluster_id)],
                    "high_position_category": category,
                    "count": int(count_table.loc[cluster_id, category]),
                    "percent_of_cluster": float(percent_table.loc[cluster_id, category]),
                }
            )
    position_crosstab_df = pd.DataFrame(crosstab_rows)

    quartile_summary_rows = []
    cluster1_subset_df = event_metrics_df.loc[event_metrics_df["cluster1_quartile_subset"].isin(["lower", "middle", "upper"])].copy()
    if not cluster1_subset_df.empty:
        for subset_name, subset in cluster1_subset_df.groupby("cluster1_quartile_subset"):
            quartile_summary_rows.append(
                {
                    "cluster1_quartile_subset": subset_name,
                    "n_events": len(subset),
                    "mean_high_center_lon_degE": float(subset["high_center_lon_degE"].mean()),
                    "mean_high_center_lat_degN": float(subset["high_center_lat_degN"].mean()),
                    "mean_high_max_msl_anomaly_hpa": float(subset["high_max_msl_anomaly_hpa"].mean()),
                    "mean_search_box_mean_msl_anomaly_hpa": float(subset["search_box_mean_msl_anomaly_hpa"].mean()),
                    "percent_events_high_center_east_of_130E": float(100.0 * subset["high_center_east_of_130E"].mean()),
                    "percent_events_high_center_south_of_55N": float(100.0 * subset["high_center_south_of_55N"].mean()),
                }
            )
    quartile_summary_df = pd.DataFrame(quartile_summary_rows)
    return cluster_summary_df, position_crosstab_df, quartile_summary_df



def draw_box(ax, box, *, edgecolor="black", linewidth=1.0, linestyle="--"):
    rect = Rectangle(
        (box.lon_min, box.lat_min),
        box.lon_max - box.lon_min,
        box.lat_max - box.lat_min,
        fill=False,
        edgecolor=edgecolor,
        linewidth=linewidth,
        linestyle=linestyle,
        transform=ccrs.PlateCarree(),
        zorder=10,
    )
    ax.add_patch(rect)



def add_map_features(ax, domain, *, title: str):
    ax.set_extent([domain.lon_min, domain.lon_max, domain.lat_min, domain.lat_max], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="50m", linewidth=0.7)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.4, edgecolor="dimgray")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 8.5}
    gl.ylabel_style = {"size": 8.5}
    ax.set_title(title, fontsize=10.2)
    draw_box(ax, SIBERIAN_HIGH_SEARCH_BOX, edgecolor="black", linewidth=1.0, linestyle="--")



def build_symmetric_levels(*arrays, n_levels: int = 13) -> np.ndarray:
    max_abs = 0.0
    for arr in arrays:
        values = np.asarray(arr.values if hasattr(arr, "values") else arr, dtype=float)
        if np.isfinite(values).any():
            max_abs = max(max_abs, float(np.nanmax(np.abs(values))))
    if not np.isfinite(max_abs) or max_abs == 0.0:
        max_abs = 1.0
    return np.linspace(-max_abs, max_abs, n_levels)



def plot_cluster_mean_maps(cluster_mean_ds: xr.Dataset, cluster_label_lookup: dict[int, str], cluster_event_counts: dict[int, int]):
    cluster_ids = sorted(cluster_event_counts)
    fields = [cluster_mean_ds["msl_anomaly_mean_hpa"].sel(cluster_id=cluster_id) for cluster_id in cluster_ids]
    levels = build_symmetric_levels(*fields)
    fig, axes = plt.subplots(1, len(cluster_ids), figsize=(6.2 * len(cluster_ids), 5.8), subplot_kw={"projection": ccrs.PlateCarree()})
    if len(cluster_ids) == 1:
        axes = [axes]
    mappable = None
    for ax, cluster_id, field in zip(axes, cluster_ids, fields):
        mappable = ax.contourf(
            field.longitude,
            field.latitude,
            field,
            levels=levels,
            cmap="RdBu_r",
            extend="both",
            transform=ccrs.PlateCarree(),
        )
        add_map_features(ax, SIBERIAN_HIGH_PLOT_DOMAIN, title=f"Cluster {cluster_id}\n{cluster_label_lookup[cluster_id]}\nMean MSL anomaly")
        ax.axvline(EAST_LONGITUDE_THRESHOLD, color="black", linewidth=0.8, linestyle=":")
        ax.axhline(SOUTH_LATITUDE_THRESHOLD, color="black", linewidth=0.8, linestyle=":")
    fig.suptitle("Eastern Siberian High composite at event peak\nMean sea-level pressure anomaly relative to NDJF monthly climatology", fontsize=12.5, y=0.98)
    fig.subplots_adjust(top=0.84, bottom=0.17, left=0.05, right=0.98, wspace=0.10)
    cax = fig.add_axes([0.18, 0.08, 0.64, 0.04])
    cbar = fig.colorbar(mappable, cax=cax, orientation="horizontal")
    cbar.set_label("MSL anomaly [hPa]")
    return fig



def plot_high_center_scatter(event_metrics_df: pd.DataFrame, cluster_label_lookup: dict[int, str]):
    cluster_colors = {1: "#1f77b4", 2: "#d95f02", 3: "#2ca02c"}
    fig, ax = plt.subplots(figsize=(8.2, 6.2), subplot_kw={"projection": ccrs.PlateCarree()})
    add_map_features(ax, SIBERIAN_HIGH_PLOT_DOMAIN, title="Event-level Eastern Siberian High centers")
    ax.axvline(EAST_LONGITUDE_THRESHOLD, color="black", linewidth=0.8, linestyle=":")
    ax.axhline(SOUTH_LATITUDE_THRESHOLD, color="black", linewidth=0.8, linestyle=":")
    for cluster_id in sorted(event_metrics_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).unique()):
        subset = event_metrics_df.loc[event_metrics_df[PRIMARY_CLUSTER_COLUMN].astype(int) == cluster_id].copy()
        sizes = 25.0 + 10.0 * np.clip(subset["high_max_msl_anomaly_hpa"].to_numpy(dtype=float), 0.0, 20.0) / 20.0
        ax.scatter(
            subset["high_center_lon_degE"],
            subset["high_center_lat_degN"],
            s=sizes,
            alpha=0.75,
            color=cluster_colors.get(cluster_id, "gray"),
            label=f"Cluster {cluster_id} (n={len(subset)})",
            transform=ccrs.PlateCarree(),
        )
    ax.legend(loc="lower left", frameon=True, fontsize=9)
    return fig



def plot_metric_boxplots(event_metrics_df: pd.DataFrame, cluster_label_lookup: dict[int, str]):
    cluster_ids = sorted(event_metrics_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).unique())
    metrics = [
        ("high_center_lon_degE", "High-center longitude [degE]"),
        ("high_center_lat_degN", "High-center latitude [degN]"),
        ("high_max_msl_anomaly_hpa", "Peak MSL anomaly [hPa]"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.8), constrained_layout=True)
    for ax, (column_name, label) in zip(axes, metrics):
        data = [event_metrics_df.loc[event_metrics_df[PRIMARY_CLUSTER_COLUMN].astype(int) == cluster_id, column_name].dropna().values for cluster_id in cluster_ids]
        ax.boxplot(data, showfliers=False, patch_artist=True, boxprops={"facecolor": "#bdbdbd"})
        ax.set_xticks(np.arange(1, len(cluster_ids) + 1))
        ax.set_xticklabels([f"C{cluster_id}" for cluster_id in cluster_ids])
        ax.set_ylabel(label)
        ax.grid(axis="y", alpha=0.25)
    fig.suptitle("Eastern Siberian High metrics by cleaned cluster", fontsize=12.5)
    return fig



def plot_position_category_bars(position_crosstab_df: pd.DataFrame):
    category_order = ["west_north", "east_north", "west_south", "east_south"]
    label_lookup = {
        "west_north": "W / N",
        "east_north": "E / N",
        "west_south": "W / S",
        "east_south": "E / S",
    }
    cluster_ids = sorted(position_crosstab_df["cluster_id"].unique().tolist())
    x = np.arange(len(category_order))
    width = 0.35 if len(cluster_ids) == 2 else 0.25
    fig, ax = plt.subplots(figsize=(8.8, 4.8))
    for offset_index, cluster_id in enumerate(cluster_ids):
        subset = position_crosstab_df.loc[position_crosstab_df["cluster_id"] == cluster_id].set_index("high_position_category").reindex(category_order)
        offset = (offset_index - (len(cluster_ids) - 1) / 2.0) * width
        ax.bar(x + offset, subset["percent_of_cluster"].values, width=width, label=f"Cluster {cluster_id}")
    ax.set_xticks(x)
    ax.set_xticklabels([label_lookup[label] for label in category_order])
    ax.set_ylabel("Percent of cluster events [%]")
    ax.set_title("Cluster occurrence by Eastern Siberian High center sector")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig


In [ ]:
paths_to_restore = [
    CLEANED_CLUSTERED_EVENTS_PATH,
    NOTEBOOK15_SOLUTION_SUMMARY_PATH,
    QUARTILE_SELECTION_PATH,
    MSL_CLIMATOLOGY_PATH,
]
for path in paths_to_restore:
    if not path.exists():
        restore_from_drive_cache(path)

if not CLEANED_CLUSTERED_EVENTS_PATH.exists():
    raise RuntimeError("Run Notebook 15 first so the cleaned clustered event table exists.")

clustered_df = pd.read_csv(CLEANED_CLUSTERED_EVENTS_PATH)
clustered_df = add_japan_local_time_columns(clustered_df)
clustered_df = clustered_df.reset_index().rename(columns={"index": "event_index"})
required_columns = [PRIMARY_CLUSTER_COLUMN, EXPLORATORY_CLUSTER_COLUMN, "event_peak", "event_peak_jst", "duration_hours", "event_index"]
missing_columns = [column for column in required_columns if column not in clustered_df.columns]
if missing_columns:
    raise RuntimeError(f"Missing required columns in the cleaned clustered event table: {missing_columns}")

cluster_counts = clustered_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).value_counts().sort_index()
cluster_label_lookup, cluster_label_df = build_cluster_labels_from_counts(cluster_counts)
clustered_df["cluster1_quartile_subset"] = "not_cluster1"
quartile_overlay_available = False
quartile_subset_counts_df = pd.DataFrame(columns=["cluster1_quartile_subset", "n_events"])

if QUARTILE_SELECTION_PATH.exists():
    quartile_selection_df = pd.read_csv(QUARTILE_SELECTION_PATH)
    if {"event_index", "quartile_subset"}.issubset(quartile_selection_df.columns):
        quartile_lookup = quartile_selection_df.drop_duplicates(subset=["event_index"]).set_index("event_index")["quartile_subset"]
        cluster1_mask = clustered_df[PRIMARY_CLUSTER_COLUMN].astype(int) == PRIMARY_CLUSTER_A_ID
        clustered_df.loc[cluster1_mask, "cluster1_quartile_subset"] = (
            clustered_df.loc[cluster1_mask, "event_index"].map(quartile_lookup).fillna("middle")
        )
        quartile_overlay_available = True
        quartile_subset_counts_df = (
            clustered_df.loc[cluster1_mask, "cluster1_quartile_subset"]
            .value_counts()
            .rename_axis("cluster1_quartile_subset")
            .reset_index(name="n_events")
            .sort_values("cluster1_quartile_subset")
            .reset_index(drop=True)
        )

climatology_years = sorted(pd.to_datetime(clustered_df["event_peak"]).dt.year.unique().tolist())
required_months = sorted(pd.to_datetime(clustered_df["event_peak"]).dt.month.unique().tolist())
era5_runtime_ds = None
msl_climatology, msl_climatology_source, era5_runtime_ds = load_or_update_msl_climatology(
    MSL_CLIMATOLOGY_PATH,
    years=climatology_years,
    months=required_months,
    domain=SIBERIAN_HIGH_PLOT_DOMAIN,
    current_ds=era5_runtime_ds,
)

climatology_summary_df = pd.DataFrame(
    {
        "metric": [
            "msl climatology source",
            "event years represented",
            "required months",
            "plot domain",
            "search box",
            "east/west threshold [degE]",
            "north/south threshold [degN]",
            "quartile overlay available",
        ],
        "value": [
            msl_climatology_source,
            f"{min(climatology_years)}-{max(climatology_years)} ({len(climatology_years)} years)",
            ", ".join(str(month) for month in required_months),
            f"{SIBERIAN_HIGH_PLOT_DOMAIN.lon_min}-{SIBERIAN_HIGH_PLOT_DOMAIN.lon_max}E, {SIBERIAN_HIGH_PLOT_DOMAIN.lat_min}-{SIBERIAN_HIGH_PLOT_DOMAIN.lat_max}N",
            f"{SIBERIAN_HIGH_SEARCH_BOX.lon_min}-{SIBERIAN_HIGH_SEARCH_BOX.lon_max}E, {SIBERIAN_HIGH_SEARCH_BOX.lat_min}-{SIBERIAN_HIGH_SEARCH_BOX.lat_max}N",
            EAST_LONGITUDE_THRESHOLD,
            SOUTH_LATITUDE_THRESHOLD,
            quartile_overlay_available,
        ],
    }
)
climatology_month_means_df = pd.DataFrame(
    {
        "month": [int(month_value) for month_value in msl_climatology["month"].values.tolist()],
        "domain_mean_hpa": [round(float(msl_climatology.sel(month=month_value).mean().values), 3) for month_value in msl_climatology["month"].values.tolist()],
    }
)

print("Cleaned cluster labels available for the Eastern Siberian High analysis")
display(cluster_label_df)
print("\nMSL climatology context")
display(climatology_summary_df)
print("\nMSL climatology domain means by month")
display(climatology_month_means_df)
if quartile_overlay_available:
    print("\nCluster 1 quartile-subset counts")
    display(quartile_subset_counts_df)


In [ ]:
required_globals = [
    "clustered_df",
    "msl_climatology",
    "quartile_overlay_available",
    "find_high_center",
    "categorize_high_position",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the setup/import cell and the Notebook 20 context/climatology cell before the one-event sanity-check cell. "
        f"Missing globals: {missing_globals}"
    )

if "era5_runtime_ds" not in globals() or era5_runtime_ds is None:
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})

sample_event_row = clustered_df.iloc[0].copy()
sample_peak = pd.Timestamp(sample_event_row["event_peak"])
sample_snapshot = load_snapshot(
    era5_runtime_ds,
    sample_peak,
    variables=[MSL_VARIABLE],
    domain=SIBERIAN_HIGH_PLOT_DOMAIN,
    level=None,
)
sample_msl_hpa = (sample_snapshot[MSL_VARIABLE] / 100.0).rename("msl_hpa_peak")
sample_anomaly = (sample_msl_hpa - msl_climatology.sel(month=sample_peak.month)).rename(FIELD_NAME)
sample_high_metrics = find_high_center(sample_anomaly, SIBERIAN_HIGH_SEARCH_BOX)
sample_position_flags = categorize_high_position(
    sample_high_metrics["high_center_lon_degE"],
    sample_high_metrics["high_center_lat_degN"],
)

sample_summary_df = pd.DataFrame(
    {
        "metric": [
            "sample event peak",
            "sample cluster",
            "sample cluster1 quartile subset",
            "sample search-box max MSL anomaly [hPa]",
            "sample high-center longitude [degE]",
            "sample high-center latitude [degN]",
            "sample search-box mean MSL anomaly [hPa]",
            "sample positive-anomaly fraction in search box",
            "sample high center east of 130E",
            "sample high center south of 55N",
            "sample high-position category",
        ],
        "value": [
            str(sample_peak),
            int(sample_event_row[PRIMARY_CLUSTER_COLUMN]),
            sample_event_row["cluster1_quartile_subset"],
            round(sample_high_metrics["high_max_msl_anomaly_hpa"], 3),
            round(sample_high_metrics["high_center_lon_degE"], 3),
            round(sample_high_metrics["high_center_lat_degN"], 3),
            round(sample_high_metrics["search_box_mean_msl_anomaly_hpa"], 3),
            round(sample_high_metrics["positive_anomaly_fraction_in_search_box"], 3),
            sample_position_flags["high_center_east_of_130E"],
            sample_position_flags["high_center_south_of_55N"],
            sample_position_flags["high_position_category"],
        ],
    }
)
print("One-event Eastern Siberian High sanity check before the full event loop")
display(sample_summary_df)
if not np.isfinite(sample_high_metrics["high_max_msl_anomaly_hpa"]):
    raise RuntimeError(
        "The sanity-check event returned a missing high-center anomaly in the Siberian High search box. "
        "Do not run the full event loop until the MSL field or search box is corrected."
    )


In [ ]:
required_globals = [
    "clustered_df",
    "cluster_label_lookup",
    "build_cluster_labels_from_counts",
    "find_high_center",
    "categorize_high_position",
    "load_cached_dataset",
    "write_checkpoint_dataset",
    "safe_concat",
    "attach_event_metadata",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the setup/import cell, the Notebook 20 context/climatology cell, and the one-event sanity-check cell before the full analysis cell. "
        f"Missing globals: {missing_globals}"
    )

for path in [
    EVENT_STACK_PATH,
    EVENT_METRICS_PATH,
    CLUSTER_COMPOSITE_PATH,
    CLUSTER_SUMMARY_PATH,
    POSITION_CROSSTAB_PATH,
    QUARTILE_HIGH_SUMMARY_PATH,
]:
    if not path.exists():
        restore_from_drive_cache(path)

reuse_final_outputs = (
    (not FORCE_REBUILD_HIGH_STACK)
    and EVENT_STACK_PATH.exists()
    and EVENT_METRICS_PATH.exists()
    and CLUSTER_COMPOSITE_PATH.exists()
    and CLUSTER_SUMMARY_PATH.exists()
    and POSITION_CROSSTAB_PATH.exists()
)

if reuse_final_outputs:
    event_stack_ds = load_cached_dataset(EVENT_STACK_PATH)
    event_metrics_df = pd.read_csv(EVENT_METRICS_PATH)
    cluster_composite_ds = load_cached_dataset(CLUSTER_COMPOSITE_PATH)
    cluster_summary_df = pd.read_csv(CLUSTER_SUMMARY_PATH)
    position_crosstab_df = pd.read_csv(POSITION_CROSSTAB_PATH)
    quartile_high_summary_df = pd.read_csv(QUARTILE_HIGH_SUMMARY_PATH) if QUARTILE_HIGH_SUMMARY_PATH.exists() else pd.DataFrame()
else:
    if not FORCE_REBUILD_HIGH_STACK:
        if not EVENT_STACK_PARTIAL_PATH.exists():
            restore_from_drive_cache(EVENT_STACK_PARTIAL_PATH)
        if not EVENT_METRICS_PARTIAL_PATH.exists():
            restore_from_drive_cache(EVENT_METRICS_PARTIAL_PATH)

    partial_metrics_df = pd.read_csv(EVENT_METRICS_PARTIAL_PATH) if (EVENT_METRICS_PARTIAL_PATH.exists() and not FORCE_REBUILD_HIGH_STACK) else pd.DataFrame()
    partial_stack_ds = load_cached_dataset(EVENT_STACK_PARTIAL_PATH) if (EVENT_STACK_PARTIAL_PATH.exists() and not FORCE_REBUILD_HIGH_STACK) else None
    processed_event_indices = set(partial_metrics_df["event_index"].astype(int).tolist()) if not partial_metrics_df.empty else set()
    pending_df = clustered_df.loc[~clustered_df["event_index"].isin(processed_event_indices)].copy().sort_values("event_peak")

    if "era5_runtime_ds" not in globals() or era5_runtime_ds is None:
        era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})

    new_metric_rows = []
    event_stack_ds = partial_stack_ds
    total_pending = len(pending_df)
    for pending_number, (_, row) in enumerate(pending_df.iterrows(), start=1):
        peak_time = pd.Timestamp(row["event_peak"])
        snapshot = load_snapshot(
            era5_runtime_ds,
            peak_time,
            variables=[MSL_VARIABLE],
            domain=SIBERIAN_HIGH_PLOT_DOMAIN,
            level=None,
        )
        msl_hpa = (snapshot[MSL_VARIABLE] / 100.0).rename("msl_hpa_peak")
        anomaly_field = (msl_hpa - msl_climatology.sel(month=peak_time.month)).rename(FIELD_NAME)
        high_metrics = find_high_center(anomaly_field, SIBERIAN_HIGH_SEARCH_BOX)
        high_position = categorize_high_position(
            high_metrics["high_center_lon_degE"],
            high_metrics["high_center_lat_degN"],
        )
        new_metric_rows.append(
            {
                "event_index": int(row["event_index"]),
                "event_peak": peak_time,
                "event_peak_jst": row["event_peak_jst"],
                "duration_hours": float(row["duration_hours"]) if pd.notna(row["duration_hours"]) else np.nan,
                PRIMARY_CLUSTER_COLUMN: int(row[PRIMARY_CLUSTER_COLUMN]),
                EXPLORATORY_CLUSTER_COLUMN: int(row[EXPLORATORY_CLUSTER_COLUMN]),
                "cluster1_quartile_subset": row["cluster1_quartile_subset"],
                "event_month": int(peak_time.month),
                **high_metrics,
                **high_position,
            }
        )
        event_stack_ds = safe_concat(event_stack_ds, attach_event_metadata(anomaly_field, row))

        if pending_number % CHECKPOINT_EVERY_EVENTS == 0 or pending_number == total_pending:
            combined_metrics_df = pd.concat([partial_metrics_df, pd.DataFrame(new_metric_rows)], ignore_index=True)
            combined_metrics_df = combined_metrics_df.drop_duplicates(subset=["event_index"], keep="last").sort_values("event_index").reset_index(drop=True)
            combined_metrics_df.to_csv(EVENT_METRICS_PARTIAL_PATH, index=False)
            maybe_copy_to_drive(EVENT_METRICS_PARTIAL_PATH)
            write_checkpoint_dataset(event_stack_ds, EVENT_STACK_PARTIAL_PATH)
            print(f"Eastern Siberian High event processing: {pending_number}/{total_pending} pending events")

    event_metrics_df = pd.concat([partial_metrics_df, pd.DataFrame(new_metric_rows)], ignore_index=True)
    event_metrics_df = event_metrics_df.drop_duplicates(subset=["event_index"], keep="last").sort_values("event_index").reset_index(drop=True)
    write_checkpoint_dataset(event_stack_ds, EVENT_STACK_PATH)
    event_metrics_df.to_csv(EVENT_METRICS_PATH, index=False)
    maybe_copy_to_drive(EVENT_METRICS_PATH)

    cluster_ids = sorted(cluster_label_lookup)
    mean_slices = []
    count_slices = []
    for cluster_id in cluster_ids:
        cluster_event_indices = event_metrics_df.loc[event_metrics_df[PRIMARY_CLUSTER_COLUMN].astype(int) == int(cluster_id), "event_index"].astype(int).tolist()
        cluster_subset = event_stack_ds.sel(event_index=cluster_event_indices)[FIELD_NAME].sortby("event_index")
        mean_slices.append(cluster_subset.mean(dim="event_index", skipna=True).expand_dims(cluster_id=[int(cluster_id)]))
        count_slices.append(cluster_subset.count(dim="event_index").astype(np.int32).expand_dims(cluster_id=[int(cluster_id)]))
    cluster_composite_ds = xr.Dataset(
        {
            "msl_anomaly_mean_hpa": xr.concat(mean_slices, dim="cluster_id"),
            "msl_anomaly_count": xr.concat(count_slices, dim="cluster_id"),
        }
    )
    cluster_composite_ds.to_netcdf(CLUSTER_COMPOSITE_PATH)
    maybe_copy_to_drive(CLUSTER_COMPOSITE_PATH)

    event_metrics_df["cluster_label"] = event_metrics_df[PRIMARY_CLUSTER_COLUMN].astype(int).map(cluster_label_lookup)
    cluster_summary_df, position_crosstab_df, quartile_high_summary_df = summarize_high_metrics(event_metrics_df, cluster_label_lookup)
    cluster_summary_df.to_csv(CLUSTER_SUMMARY_PATH, index=False)
    position_crosstab_df.to_csv(POSITION_CROSSTAB_PATH, index=False)
    maybe_copy_to_drive(CLUSTER_SUMMARY_PATH)
    maybe_copy_to_drive(POSITION_CROSSTAB_PATH)
    if not quartile_high_summary_df.empty:
        quartile_high_summary_df.to_csv(QUARTILE_HIGH_SUMMARY_PATH, index=False)
        maybe_copy_to_drive(QUARTILE_HIGH_SUMMARY_PATH)

cluster_event_counts = event_metrics_df[PRIMARY_CLUSTER_COLUMN].astype(int).value_counts().sort_index().to_dict()
monthly_cluster_counts_df = (
    event_metrics_df.groupby([PRIMARY_CLUSTER_COLUMN, "event_month"]).size().rename("n_events").reset_index()
)
print("Eastern Siberian High cluster summary")
display(cluster_summary_df)
print("\nEastern Siberian High position-category summary")
display(position_crosstab_df)
if not quartile_high_summary_df.empty:
    print("\nCluster 1 quartile-subset Eastern Siberian High summary")
    display(quartile_high_summary_df)


In [ ]:
required_globals = [
    "cluster_composite_ds",
    "event_metrics_df",
    "position_crosstab_df",
    "cluster_label_lookup",
    "cluster_event_counts",
    "plot_cluster_mean_maps",
    "plot_high_center_scatter",
    "plot_metric_boxplots",
    "plot_position_category_bars",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the Notebook 20 full analysis cell before the plotting cell. "
        f"Missing globals: {missing_globals}"
    )

plot_records = []

cluster_map_fig = plot_cluster_mean_maps(cluster_composite_ds, cluster_label_lookup, cluster_event_counts)
cluster_map_path = PLOT_DIR / "cleaned_k2_siberian_high_cluster_mean_maps.png"
cluster_map_drive = None
if SAVE_PLOTS:
    cluster_map_fig.savefig(cluster_map_path, dpi=180, bbox_inches="tight")
    cluster_map_drive = maybe_copy_to_drive(cluster_map_path, verbose=False)
plot_records.append({"plot_kind": "cluster_mean_maps", "local_path": str(cluster_map_path), "drive_path": str(cluster_map_drive) if cluster_map_drive else ""})
plt.show(cluster_map_fig)

scatter_fig = plot_high_center_scatter(event_metrics_df, cluster_label_lookup)
scatter_path = PLOT_DIR / "cleaned_k2_siberian_high_center_scatter.png"
scatter_drive = None
if SAVE_PLOTS:
    scatter_fig.savefig(scatter_path, dpi=180, bbox_inches="tight")
    scatter_drive = maybe_copy_to_drive(scatter_path, verbose=False)
plot_records.append({"plot_kind": "high_center_scatter", "local_path": str(scatter_path), "drive_path": str(scatter_drive) if scatter_drive else ""})
plt.show(scatter_fig)

boxplot_fig = plot_metric_boxplots(event_metrics_df, cluster_label_lookup)
boxplot_path = PLOT_DIR / "cleaned_k2_siberian_high_metric_boxplots.png"
boxplot_drive = None
if SAVE_PLOTS:
    boxplot_fig.savefig(boxplot_path, dpi=180, bbox_inches="tight")
    boxplot_drive = maybe_copy_to_drive(boxplot_path, verbose=False)
plot_records.append({"plot_kind": "metric_boxplots", "local_path": str(boxplot_path), "drive_path": str(boxplot_drive) if boxplot_drive else ""})
plt.show(boxplot_fig)

position_fig = plot_position_category_bars(position_crosstab_df)
position_path = PLOT_DIR / "cleaned_k2_siberian_high_position_category_bars.png"
position_drive = None
if SAVE_PLOTS:
    position_fig.savefig(position_path, dpi=180, bbox_inches="tight")
    position_drive = maybe_copy_to_drive(position_path, verbose=False)
plot_records.append({"plot_kind": "position_category_bars", "local_path": str(position_path), "drive_path": str(position_drive) if position_drive else ""})
plt.show(position_fig)

plot_inventory_df = pd.DataFrame(plot_records)
plot_inventory_df.to_csv(PLOT_INVENTORY_PATH, index=False)
maybe_copy_to_drive(PLOT_INVENTORY_PATH)
print("Saved Eastern Siberian High plot inventory")
display(plot_inventory_df)


In [ ]:
required_globals = [
    "cluster_label_df",
    "cluster_summary_df",
    "position_crosstab_df",
    "quartile_high_summary_df",
    "climatology_summary_df",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the Notebook 20 context, analysis, and plotting cells before the summary cell. "
        f"Missing globals: {missing_globals}"
    )

print("Notebook 20 Eastern Siberian High summary")
display(climatology_summary_df)
print("\nCleaned cluster counts")
display(cluster_label_df)
print("\nCluster-wise Eastern Siberian High summary")
display(cluster_summary_df)
print("\nCluster occurrence by Eastern Siberian High center sector")
display(position_crosstab_df)
if not quartile_high_summary_df.empty:
    print("\nCluster 1 quartile-overlay Eastern Siberian High summary")
    display(quartile_high_summary_df)
print(
    "\nNotebook 20 now does the following:\n"
    "- quantifies Eastern Siberian High event-peak position using the maximum positive MSL anomaly inside a broad Siberian search box\n"
    "- compares cleaned Cluster 1 and Cluster 2 by high-center longitude, latitude, anomaly magnitude, and sector occupancy\n"
    "- uses Notebook 18 quartile tags as an optional Cluster 1 overlay when they are available\n"
    "- saves event metrics, monthly climatology, cluster composites, and plot files for the next interpretation step"
)
